# 05 - Pupil & LHIPA

**Audience:** computing cognitive-load indices on pupil data.

LHIPA (Duchowski et al. 2018) is a wavelet-decomposition metric: ratio of
low- to high-frequency pupil oscillation power. Higher LHIPA = higher
cognitive load. The Python implementation matches the on-device live
monitor in Unity (`RealtimeCognitiveLoad.cs`) byte-for-byte.

**Requires** `PyWavelets` - install with `pip install -e .[wavelet]`.

## What you'll get

- Raw pupil signal + cleaned signal (NaN-mask blinks).
- LHIPA across the whole recording.
- LHIPA in sliding windows so you can see load over time.
- Comparison against an early-recording baseline.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Pull pupil signal + handle blinks

In [ ]:
df = ctx.data.df
ts = ctx.data.get_timestamps()
if not ("left_pupil_diameter" in df.columns or "right_pupil_diameter" in df.columns):
    raise RuntimeError("No pupil columns in this CSV.")
left = df["left_pupil_diameter"].to_numpy(dtype=float) if "left_pupil_diameter" in df.columns else None
right = df["right_pupil_diameter"].to_numpy(dtype=float) if "right_pupil_diameter" in df.columns else None
if left is not None and right is not None:
    pupil = np.nanmean([left, right], axis=0)
elif left is not None:
    pupil = left
else:
    pupil = right
# Mask physiologically impossible diameters (sensor dropouts) to NaN.
mask = (pupil < 1.0) | (pupil > 9.0)
pupil_clean = pupil.copy()
pupil_clean[mask] = np.nan
print(f"Total samples: {len(pupil)}, NaN-masked: {int(mask.sum())} ({mask.mean()*100:.1f}%)")
sr = ctx.data.get_sample_rate() or 90.0
print(f"Sample rate:   {sr:.1f} Hz")

## 2. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(ts, pupil, color="silver", linewidth=0.6, label="raw")
ax.plot(ts, pupil_clean, color="steelblue", linewidth=0.7, label="masked")
ax.set_xlabel("time (s)")
ax.set_ylabel("pupil (mm)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Whole-recording LHIPA

In [ ]:
res = ela.calculate_lhipa(pupil_clean, sample_rate=sr)
print(f"LHIPA: {res.lhipa:.4f}")
print(f"  valid:   {res.is_valid}")
if not res.is_valid:
    print(f"  reason:  {res.error_message}")
print(f"  low IPA:  {res.low_ipa:.4f}")
print(f"  high IPA: {res.high_ipa:.4f}")
print(f"  n samples used: {res.n_samples}")

## 4. Sliding-window LHIPA

Computed per ~10-second window; gives you a load timeline.

In [ ]:
win_s = 10.0
step_s = 5.0
win_n = int(win_s * sr)
step_n = int(step_s * sr)
centers = []
vals = []
i = 0
while i + win_n <= len(pupil_clean):
    chunk = pupil_clean[i:i + win_n]
    r = ela.calculate_lhipa(chunk, sample_rate=sr)
    centers.append(ts[i + win_n // 2])
    vals.append(r.lhipa if r.is_valid else np.nan)
    i += step_n
if centers:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(centers, vals, marker="o", linewidth=1.0)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("LHIPA")
    ax.set_title(f"Sliding LHIPA  ({win_s:.0f}s window, {step_s:.0f}s step)")
    plt.tight_layout()
    plt.show()
else:
    print("Recording too short for sliding LHIPA.")

## 5. Baseline comparison

Compute LHIPA on the first 30 s as a relaxed baseline and compare to
the rest of the recording.

In [ ]:
baseline_n = int(min(30.0 * sr, len(pupil_clean) // 2))
if baseline_n < int(5 * sr):
    print("Recording too short for a baseline comparison.")
else:
    base = ela.calculate_lhipa(pupil_clean[:baseline_n], sample_rate=sr)
    rest = ela.calculate_lhipa(pupil_clean[baseline_n:], sample_rate=sr)
    print(f"Baseline (first {baseline_n/sr:.0f}s):  LHIPA={base.lhipa:.4f}  valid={base.is_valid}")
    print(f"Rest of recording:        LHIPA={rest.lhipa:.4f}  valid={rest.is_valid}")
    if base.is_valid and rest.is_valid:
        delta = rest.lhipa - base.lhipa
        direction = 'higher load' if delta > 0 else 'lower load'
        print(f"\nDelta: {delta:+.4f} ({direction} relative to baseline)")